# 06 — Erreurs des prévisions météo (brouillon)

Vraies prévisions ECMWF passées (API Single Runs, une par jour à 0 h UTC, heure par heure sur 10 jours) face à la météo observée (ERA5-Land), sur les carreaux de La Clapière. But : voir à quoi ressemblent les erreurs avant de choisir comment abîmer la météo future à l'entraînement.

## Cadre : notations, hypothèses, objectifs

**Notations**
- $r$ : instant d'émission d'une prévision (une par jour, 0 h UTC) ; $t$ : échéance (heures ou jours après $r$) ; $\Delta$ : pas du cumul (durée).
- $O$ : la « vérité » (ERA5-Land) ; $P$ : la prévision (ECMWF IFS 9 km, archive Open-Meteo). Pour la pluie : cumuls sur $]r+t-\Delta,\; r+t]$ (mm) ; pour la température : moyennes sur la même fenêtre (°C).
- « Sec » : moins de $0{,}1$ mm. $p_0$ : probabilité de sec ; $\mu$, $k$, $\theta = \mu/k$ : moyenne, forme et échelle de la gamma de la partie non sèche ; $\ell = \ln o$.
- $\varepsilon = T_P - T_O$ : erreur de température ; $\sigma$ : son écart-type.
- $X$ : le tableau d'entrées du modèle de débit (`Features.X`), dont la météo future est faite de cumuls par pas.

**Hypothèses**
1. ERA5-Land est la vérité de la météo passée : le modèle de débit apprend dessus.
2. **On fait confiance à Open-Meteo : la prévision est l'espérance de la vérité.** Pour toute variable $V$, toute échéance $t$ et toute valeur prévue $p$ :
$$
\forall V,\;\; \forall t,\;\; \forall p : \qquad \mathbb{E}\big[\,V_O \;\big|\; V_P = p,\; t\,\big] = p .
$$
Aucune correction de biais propre à notre bassin (un seul bassin ne permet pas de les estimer de façon générale). Conséquences : la vérité s'écrit $V_O = V_P + \eta$ avec $\mathbb{E}[\eta \mid V_P] = 0$ ; dans l'autre sens, $\mathbb{E}[V_P \mid V_O]$ n'est **pas** $V_O$ (régression vers la moyenne), elle s'obtient par la formule de Bayes. Pour la température, l'erreur est très petite devant ses variations, si bien que $T_P \approx T_O + \varepsilon$ avec $\mathbb{E}[\varepsilon] \approx 0$ (pente mesurée 1,00).
   Exception : une prévision de pluie « sèche » ($p < 0{,}1$ mm, valeur arrondie) ne peut pas imposer $\mathbb{E}[O] = 0$ : il y a de vraies pluies ratées. Pour elle, la loi de $O$ est laissée libre (fonction de $t$ seulement).
3. Les lois d'erreur ne dépendent que de l'échéance (et de la saison pour la température), pas du lieu : elles doivent valoir partout.
4. Pas de 24 h pour la pluie et la température futures dans $X$.
5. Neige au sol et humidité du sol : seulement à l'instant de la prévision dans $X$, sans futur et sans perturbation pour l'instant. Évapotranspiration future : à tester sans.
6. Une seule prévision par jour (0 h UTC) ; la latence de publication est ignorée pour l'instant.

**Objectifs**
1. Pour chaque variable, une loi de $P$ sachant $O$ et l'échéance, pour fabriquer à partir d'ERA5 des prévisions aussi imparfaites que les vraies, et entraîner le modèle de débit dessus.
2. Des lois simples, avec peu de paramètres, valables ailleurs qu'à La Clapière.
3. Vérifiées sur une année jamais vue : histogrammes, calibration (PIT), fiabilité de la probabilité de sec.
4. Ensuite : la corrélation des erreurs entre carreaux voisins et d'un jour à l'autre.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_racine = Path.cwd()
while not (_racine / "src").exists() and _racine != _racine.parent:
    _racine = _racine.parent
if str(_racine) not in sys.path:
    sys.path.insert(0, str(_racine))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config

RUNS = config.CACHE_PREVISION / "single_runs" / "ecmwf_ifs" / "runs"
OBS = config.CACHE_METEO / "carreaux"

## 1. Charger prévisions et observations

`prev` : une ligne par (carreau, prévision émise à `run`, heure prévue `time`), avec `echeance` = heures entre l'émission et l'heure prévue. `obs` : la météo observée des mêmes carreaux, heure par heure. Pour l'instant : la pluie seulement.

In [ ]:
VARIABLE = "precipitation"

prev = pd.read_parquet(RUNS, columns=["carreau", "run", "time", VARIABLE])
prev["echeance"] = ((prev["time"] - prev["run"]) / pd.Timedelta("1h")).astype(int)
carreaux = sorted(prev["carreau"].unique())

obs = pd.concat({c: pd.read_parquet(OBS / f"{c}_horaire.parquet", columns=[VARIABLE])[VARIABLE] for c in carreaux},
                names=["carreau", "time"]).rename(f"{VARIABLE}_obs")

prev = prev.join(obs, on=["carreau", "time"]).dropna(subset=[VARIABLE, f"{VARIABLE}_obs"])
print(f"{len(carreaux)} carreaux, {prev['run'].nunique()} prévisions, {len(prev):,} heures prévues comparables")
prev.head()

## 2. Pluie prévue 3 jours avant, contre pluie tombée (heure par heure)

Une heure d'un carreau = un point : la pluie de l'heure qui finit `ECHEANCE` heures après l'émission, prévue (abscisse) et tombée (ordonnée). Diagonale = prévision parfaite. « Sec » = moins de 0,1 mm.

In [ ]:
ECHEANCE = 72          # heures entre l'émission de la prévision et l'heure prévue
d = prev[prev["echeance"] == ECHEANCE]
x, y = d[VARIABLE], d[f"{VARIABLE}_obs"]
haut = max(x.max(), y.max())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 5.5))
fig.suptitle(f"Pluie horaire prévue {ECHEANCE} h à l'avance contre pluie tombée — {len(d):,} heures, {d['carreau'].nunique()} carreaux")
a1.scatter(x, y, s=3, alpha=0.25, label="une heure d'un carreau")
a1.plot([0, haut], [0, haut], "k--", lw=1, label="prévision parfaite (prévu = tombé)")
a1.set_xlabel(f"pluie prévue {ECHEANCE} h avant (mm dans l'heure)"); a1.set_ylabel("pluie tombée, ERA5-Land (mm dans l'heure)")
a1.set_title("échelles normales"); a1.legend()

a2.scatter(x + 0.01, y + 0.01, s=3, alpha=0.25)            # + 0,01 mm pour voir les zéros sur une échelle log
a2.plot([0.01, haut], [0.01, haut], "k--", lw=1)
a2.set_xscale("log"); a2.set_yscale("log")
a2.set_xlabel("pluie prévue + 0,01 (mm dans l'heure, échelle log)"); a2.set_ylabel("pluie tombée + 0,01 (mm dans l'heure, échelle log)")
a2.set_title("échelles logarithmiques : colonne de gauche = pluies ratées, ligne du bas = fausses alertes")
plt.tight_layout(); plt.show()

sec_prevu, sec_tombe = x < 0.1, y < 0.1                     # « sec » : moins de 0,1 mm dans l'heure
print(f"corrélation : {np.corrcoef(x, y)[0, 1]:.3f}   |   moyenne prévue {x.mean():.3f}, tombée {y.mean():.3f} mm/h")
pd.crosstab(np.where(sec_prevu, "prévu sec", "prévu pluie"), np.where(sec_tombe, "tombé sec", "tombé pluie"),
            normalize=True, rownames=["prévision"], colnames=["réalité"]).round(3)

Même chose en densité : couleur = nombre d'heures par case (échelle log). Les bandes vides à droite viennent de l'arrondi à 0,1 mm.

In [ ]:
from matplotlib.colors import LogNorm

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle(f"Densité des couples (pluie prévue {ECHEANCE} h avant, pluie tombée) — {len(d):,} heures")
cases = np.arange(0, haut + 0.2, 0.2)                                     # cases de 0,2 mm × 0,2 mm
_, _, _, im = a1.hist2d(x, y, bins=[cases, cases], norm=LogNorm(), cmap="turbo")
a1.plot([0, haut], [0, haut], "k--", lw=1, label="prévision parfaite"); a1.legend(loc="upper left")
a1.set_xlabel(f"pluie prévue {ECHEANCE} h avant (mm dans l'heure)"); a1.set_ylabel("pluie tombée (mm dans l'heure)")
a1.set_title("cases de 0,2 mm")
fig.colorbar(im, ax=a1, label="nombre d'heures dans la case (échelle log)")

cases_log = np.logspace(-2, np.log10(haut + 0.01), 30)                    # cases régulières en échelle log
_, _, _, im = a2.hist2d(x + 0.01, y + 0.01, bins=[cases_log, cases_log], norm=LogNorm(), cmap="turbo")
a2.plot([0.01, haut], [0.01, haut], "k--", lw=1)
a2.set_xscale("log"); a2.set_yscale("log")
a2.set_xlabel("pluie prévue + 0,01 (mm, échelle log)"); a2.set_ylabel("pluie tombée + 0,01 (mm, échelle log)")
a2.set_title("cases régulières en échelle log")
fig.colorbar(im, ax=a2, label="nombre d'heures dans la case (échelle log)")
plt.tight_layout(); plt.show()

## 3. Lois conditionnelles, cumuls de 24 h

On fixe une classe de la **condition** et on regarde la loi de l'autre variable. `CONDITION = "tombe"` : loi de la prévision $P$ sachant la pluie tombée $O$ — c'est celle qui sert à fabriquer des prévisions à partir d'ERA5 ; `"prevu"` : l'inverse.

La loi a deux morceaux, estimés sur tous les cas de la classe :
$$
\mathbb{P}(Y \le y \mid c) = p_0(c) + \big(1-p_0(c)\big)\,F_\Gamma\big(y\,;k(c),\theta(c)\big)
$$
- $p_0(c)$ = part des cas secs (« % secs » dans les titres) ;
- $k, \theta$ = gamma ajustée sur les cas non secs, par maximum de vraisemblance : $\hat\theta = \bar x/\hat k$ et $\ln \hat k - \psi(\hat k) = \ln \bar x - \overline{\ln x}$ (`scipy.stats.gamma.fit(x, floc=0)`). Une gamma ne peut pas valoir 0, d'où la séparation des cas secs.

Dans le tableau, `moyenne` et `ecart_type` sont ceux des **cas non secs** ($k\theta$ et $\sqrt{k}\,\theta$) ; `moyenne_totale` compte les cas secs : $\big(1-p_0\big)\,k\theta$.

In [ ]:
from scipy import stats

CONDITION = "tombe"                 # "tombe" : prévision sachant la pluie tombée ; "prevu" : pluie tombée sachant la prévision
SEC = 0.1                           # en dessous : « sec » (mm)


def noms(condition):
    autre = "tombe" if condition == "prevu" else "prevu"
    return autre, {"prevu": f"pluie prévue {ECHEANCE} h avant", "tombe": "pluie tombée"}


def cumuls(cumul):
    """Pluie prévue et tombée cumulées sur les `cumul` heures qui finissent à l'échéance, par (carreau, prévision)."""
    fen = prev[(prev["echeance"] > ECHEANCE - cumul) & (prev["echeance"] <= ECHEANCE)]
    g = fen.groupby(["carreau", "run"]).agg(prevu=(VARIABLE, "sum"), tombe=(f"{VARIABLE}_obs", "sum"), n=(VARIABLE, "size"))
    g = g[g["n"] == cumul]                               # seulement les cumuls complets
    print(f"{len(g):,} cumuls de {cumul} h finissant {ECHEANCE} h après l'émission")
    return g


def lois_par_classe(g, condition, classes, cumul):
    """9 histogrammes (case « sec » + 8 classes) avec la gamma ajustée ; renvoie le tableau des paramètres."""
    autre, nom = noms(condition)
    fig, axes = plt.subplots(3, 3, figsize=(15, 12)); axes = axes.ravel()
    fig.suptitle(f"Loi de la {nom[autre]} sachant la {nom[condition]} (cumuls de {cumul} h) : histogramme et gamma ajustée", fontsize=13)
    lignes = []
    for ax, (a, b) in zip(axes, [(0, SEC)] + list(zip(classes[:-1], classes[1:]))):
        v = g.loc[(g[condition] >= a) & (g[condition] < b), autre]
        seche = a == 0
        libelle = f"{nom[condition]} : sec (< {SEC} mm)" if seche else f"{nom[condition]} entre {a} et {b} mm"
        pluie = v[v >= SEC]
        if len(pluie) < 20:
            ax.set_title(f"{libelle} : trop peu de cas ({len(v)})"); continue
        k, _, theta = stats.gamma.fit(pluie, floc=0)
        xs = np.linspace(SEC, pluie.quantile(0.99), 200)
        ax.hist(pluie, bins=40, range=(0, pluie.quantile(0.99)), density=True, alpha=0.5, label=f"{len(pluie)} cas non secs")
        ax.plot(xs, stats.gamma.pdf(xs, k, scale=theta), "C3", lw=2, label=f"gamma ajustée : k = {k:.2f}, θ = {theta:.2f} mm")
        ax.axvline(pluie.mean(), color="k", ls=":", lw=1, label=f"moyenne des non secs {pluie.mean():.2f} mm")
        ax.set_title(f"{libelle}\n{len(v)} cas, dont {100 * (v < SEC).mean():.1f} % secs (masse en zéro)", fontsize=10)
        ax.set_xlabel(f"{nom[autre]} (mm sur {cumul} h)"); ax.set_ylabel("densité de probabilité (1/mm)")
        ax.legend(fontsize=8)
        lignes.append({"classe (mm)": "sec" if seche else f"{a}–{b}", "seche": seche, "cas": len(v),
                       "part_sec": (v < SEC).mean(), "moyenne": pluie.mean(), "ecart_type": pluie.std(),
                       "moyenne_totale": v.mean(), "k": k, "theta": theta})
    plt.tight_layout(); plt.show()
    return pd.DataFrame(lignes).set_index("classe (mm)")


def dispersion(tab, condition, cumul):
    """Écart-type contre moyenne des cas non secs (axes log) et droite σ = A μ^p ; renvoie (A, p)."""
    autre, nom = noms(condition)
    tab = tab[~tab["seche"]]                                         # la case « sec » est un autre régime
    lm, ls = np.log(tab["moyenne"]), np.log(tab["ecart_type"])
    p, ln_A = np.polyfit(lm, ls, 1)
    A, r2 = np.exp(ln_A), np.corrcoef(lm, ls)[0, 1] ** 2
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.loglog(tab["moyenne"], tab["ecart_type"], "o", ms=8, label="une classe (cas non secs)")
    m = np.geomspace(tab["moyenne"].min(), tab["moyenne"].max(), 50)
    ax.loglog(m, A * m ** p, "C3", lw=2, label=f"droite ajustée : σ = {A:.2f} · μ^{p:.2f}   (R² = {r2:.3f})")
    ax.loglog(m, tab["ecart_type"].iloc[0] * m / m[0], "k--", lw=1, label="pente 1 : σ ∝ μ (k constant)")
    ax.loglog(m, tab["ecart_type"].iloc[0] * np.sqrt(m / m[0]), "k:", lw=1, label="pente 1/2 : σ² ∝ μ (θ constant)")
    ax.set_xlabel(f"moyenne μ de la {nom[autre]}, cas non secs (mm sur {cumul} h, échelle log)")
    ax.set_ylabel("écart-type σ, cas non secs (mm, échelle log)")
    ax.set_title(f"Dispersion contre moyenne (cumuls de {cumul} h)"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
    print(f"σ(μ) = {A:.3f} · μ^{p:.3f}   →   k(μ) = μ^{2 - 2 * p:.3f} / {A ** 2:.3f},   θ(μ) = {A ** 2:.3f} · μ^{2 * p - 1:.3f}")
    return A, p


def esperance(g, condition, cumul, n_classes=20):
    """Moyenne de l'autre variable (secs compris) par classe de même effectif de la condition, et loi de puissance a c^b."""
    autre, nom = noms(condition)
    pluvieux = g[g[condition] >= SEC]
    bornes = np.unique(np.quantile(pluvieux[condition], np.linspace(0, 1, n_classes + 1)))
    groupe = pluvieux.groupby(pd.cut(pluvieux[condition], bornes, include_lowest=True), observed=True)
    tab = pd.DataFrame({"condition": groupe[condition].mean(), "moyenne": groupe[autre].mean(),
                        "erreur": 2 * groupe[autre].std() / np.sqrt(groupe.size()), "cas": groupe.size()})
    b, ln_a = np.polyfit(np.log(tab["condition"]), np.log(tab["moyenne"]), 1)
    a = np.exp(ln_a)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))
    fig.suptitle(f"Espérance de la {nom[autre]} sachant la {nom[condition]} (cumuls de {cumul} h, classes de même effectif)")
    for ax, log in ((a1, False), (a2, True)):
        ax.errorbar(tab["condition"], tab["moyenne"], yerr=tab["erreur"], fmt="o", capsize=3, label="moyenne par classe ± 2 erreurs-types")
        lim = [tab["condition"].min(), tab["condition"].max()]
        cc = np.geomspace(*lim, 50)
        ax.plot(lim, lim, "k--", lw=1, label="E[autre | c] = c (sans biais)")
        ax.plot(cc, a * cc ** b, "C3", lw=2, label=f"ajustement : E = {a:.2f} · c^{b:.2f}")
        if log:
            ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(f"{nom[condition]}, moyenne de la classe (mm sur {cumul} h{', échelle log' if log else ''})")
        ax.set_ylabel(f"moyenne de la {nom[autre]}, cas secs compris (mm)"); ax.legend(fontsize=8)
    a1.set_title("échelles normales"); a2.set_title("échelles logarithmiques")
    plt.tight_layout(); plt.show()
    print(f"E[{autre} | {condition} = c] ≈ {a:.3f} · c^{b:.3f}   |   moyennes globales : {nom[condition]} {g[condition].mean():.3f}, {nom[autre]} {g[autre].mean():.3f} mm")
    return a, b, tab


g24 = cumuls(24)
lois24 = lois_par_classe(g24, CONDITION, [0.1, 0.25, 0.5, 1, 2, 4, 8, 16, 32], 24)
lois24.round(3)

**Dispersion.** Avec $\mu = k\theta$ et $\sigma = \sqrt{k}\,\theta$ : pente 1 en log-log ⟺ $k$ constant ; pente 1/2 ⟺ $\theta$ constant. La droite $\sigma = A\mu^p$ donne $k(\mu) = \mu^{2-2p}/A^2$ et $\theta(\mu) = A^2\mu^{2p-1}$.

In [ ]:
A24, p24 = dispersion(lois24, CONDITION, 24)

**Biais conditionnel.** Peut-on centrer la loi sur la valeur de la condition, $\mathbb{E}[\text{autre} \mid c] = c$ ? Si les points suivent la diagonale, oui ; sinon on garde l'ajustement $a\,c^{\,b}$. Les moyennes comptent les cas secs.

In [ ]:
a24, b24, esp24 = esperance(g24, CONDITION, 24)
esp24.round(2)

## 4. Même chose heure par heure

La pluie de l'heure qui finit pile à l'échéance. L'écart avec les cumuls de 24 h mesure l'erreur de **timing** : une averse décalée de deux heures est une double erreur à l'heure, aucune sur 24 h.

In [ ]:
g1 = cumuls(1)
lois1 = lois_par_classe(g1, CONDITION, [0.1, 0.2, 0.3, 0.5, 1, 2, 3, 5, 8], 1)
lois1.round(3)

In [ ]:
A1, p1 = dispersion(lois1, CONDITION, 1)

In [ ]:
a1_, b1_, esp1 = esperance(g1, CONDITION, 1)
esp1.round(2)

## 5. Erreurs de timing : corrélation décalée

Pour une échéance $h$, corrélation entre la pluie horaire **prévue** à $r+h$ et la pluie **tombée** à $r+h+j$, pour $j$ de −24 h à +24 h (tous carreaux) :
$$
C_h(j) = \operatorname{corr}\big(P_{r+h},\, O_{r+h+j}\big).
$$
Référence « prévision parfaite » : $\operatorname{corr}(O_t, O_{t+j})$ — une averse dure plusieurs heures, donc même parfaite la courbe est une cloche. L'erreur de timing est ce qui dépasse la référence : pic plus bas, cloche plus large. Un pic décalé de $j=0$ serait un retard ou une avance systématique.

In [ ]:
ECHEANCES_TIMING = [24, 72, 120, 168]          # 1, 3, 5, 7 jours
DECALAGES = np.arange(-24, 25)

def correlation_decalee(paires_x, serie_y, decalages):
    """corr(x, y décalée de j heures) pour chaque j ; x indexé par (carreau, time), y série horaire indexée pareil."""
    out = []
    for j in decalages:
        cles = pd.MultiIndex.from_arrays([paires_x.index.get_level_values(0),
                                          paires_x.index.get_level_values(1) + pd.Timedelta(hours=int(j))])
        y = serie_y.reindex(cles).to_numpy()
        ok = ~np.isnan(y)
        out.append(np.corrcoef(paires_x.to_numpy()[ok], y[ok])[0, 1])
    return np.array(out)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Corrélation entre la pluie horaire prévue à l'échéance et la pluie tombée décalée de j heures")
ref = None
largeurs = {}
for h, couleur in zip(ECHEANCES_TIMING, ["C0", "C1", "C2", "C3"]):
    d = prev[prev["echeance"] == h].set_index(["carreau", "time"])[VARIABLE]
    cj = correlation_decalee(d, obs, DECALAGES)
    if ref is None:                                              # référence : la pluie tombée contre elle-même
        o = obs.reindex(d.index)
        ref = correlation_decalee(o.dropna(), obs, DECALAGES)
        a1.plot(DECALAGES, ref, "k--", lw=1.5, label="prévision parfaite (pluie tombée contre elle-même)")
        a2.plot(DECALAGES, ref / ref.max(), "k--", lw=1.5, label="prévision parfaite")
    a1.plot(DECALAGES, cj, "o-", ms=3, color=couleur, label=f"échéance {h} h ({h // 24} j) : pic {cj.max():.2f} à j = {DECALAGES[cj.argmax()]} h")
    a2.plot(DECALAGES, cj / cj.max(), "o-", ms=3, color=couleur, label=f"{h // 24} j")
    largeurs[h] = (cj >= cj.max() / 2).sum()                     # largeur à mi-hauteur, en heures
largeurs["parfaite"] = (ref >= ref.max() / 2).sum()
a1.set_xlabel("décalage j (heures) : pluie tombée à échéance + j"); a1.set_ylabel("corrélation")
a1.set_title("corrélation brute"); a1.axvline(0, color="grey", lw=0.5); a1.legend(fontsize=8)
a2.set_xlabel("décalage j (heures)"); a2.set_ylabel("corrélation / corrélation au pic")
a2.set_title("même chose ramenée à 1 au pic : comparer les largeurs"); a2.axvline(0, color="grey", lw=0.5); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()
pd.Series(largeurs, name="largeur à mi-hauteur (h)")

## 6. Conclusion (pluie, prévision à 3 jours, La Clapière, mars 2024 – octobre 2026)

**Sur 24 h, la prévision sachant la pluie tombée $o$ se décrit bien par deux morceaux :**
1. **sec** avec une probabilité $p_0(o)$ qui décroît avec $o$ : 89 % si rien n'est tombé (11 % de fausses alertes, petites : 0,8 mm en moyenne), 22 % pour 0,5–1 mm, 3 % pour 4–8 mm, 0 % au-delà de 16 mm ;
2. sinon une **gamma** dont la forme et l'échelle grandissent avec la pluie ($k$ de 0,8 à 3 : presque exponentielle pour les petites pluies, en bosse pour les fortes), avec $\sigma \approx 1{,}56\,\mu^{0{,}61}$ entre écart-type et moyenne.

**La pluie tombée n'est pas la bonne espérance** : $\mathbb{E}[P \mid O = o] \approx 1{,}67\,o^{0{,}70}$. La prévision surestime les petites pluies (0,6 mm prévu pour 0,14 mm tombé), est juste vers 4–5 mm, et **sous-estime les fortes** (24 mm prévus pour 30 mm tombés) — ce qui compte pour les crues.

**Ce qu'on prendra donc pour fabriquer une prévision de pluie à partir d'ERA5** (cumul $o$ sur le pas) :
$$
P = \begin{cases} 0 & \text{avec la probabilité } p_0(o) \\ \Gamma\big(k(m), \theta(m)\big) & \text{sinon, avec } m = \dfrac{\mathbb{E}[P \mid O=o]}{1-p_0(o)},\; k = \dfrac{m^{2-2p}}{A^2},\; \theta = A^2 m^{2p-1} \end{cases}
$$
et **pas** une perturbation centrée sur $o$.

**Heure par heure, c'est bien pire** : même quand il est tombé 3 à 5 mm dans l'heure, la prévision disait sec 13 % du temps, et elle ne donne en moyenne que la moitié des fortes pluies horaires ($\mathbb{E}[P \mid O=o] \approx 0{,}78\,o^{0{,}60}$). C'est l'erreur de **timing** : l'averse est prévue, mais pas à la bonne heure. Il faut donc perturber les **cumuls sur le pas** de X (6 à 12 h), pas les heures une à une.

**À faire ensuite :** refaire ces mesures au pas de X (12 h) et pour plusieurs échéances (1 à 9 jours) ; vérifier que l'erreur est la même pour les carreaux voisins (un même épisode) ; passer aux autres variables (température, neige).

## 7. Toutes les échéances : la dépendance en $t$

Même pas de temps qu'en partie 3 : **cumuls de 24 h** (c'est le pas prévu pour la pluie future dans le tableau d'entrées du modèle). On prend les fenêtres $]t-24\text{ h}, t]$ pour $t$ = 1, 2, … 9 jours, tous les carreaux. On garde la dernière année (prévisions émises à partir du 1er octobre 2025) pour la **vérification** de la partie 9 : le modèle ne la voit pas.

In [ ]:
PAS_MODELE = 24
COUPURE = pd.Timestamp("2025-10-01", tz="UTC")          # prévisions émises après : jeu de vérification

def cumuls_echeance(t, pas=PAS_MODELE):
    f = prev[(prev["echeance"] > t - pas) & (prev["echeance"] <= t)]
    g = f.groupby(["carreau", "run"]).agg(p=(VARIABLE, "sum"), o=(f"{VARIABLE}_obs", "sum"), n=(VARIABLE, "size"))
    return g[g["n"] == pas].reset_index().assign(t=t)

D = pd.concat([cumuls_echeance(t) for t in range(24, 240, 24)], ignore_index=True)
D["jours"] = D["t"] / 24
D["test"] = D["run"] >= COUPURE
print(f"{len(D):,} cumuls de {PAS_MODELE} h ; {D['test'].mean():.1%} gardés pour la vérification")

## 8. Le modèle : $p_0(o,t)$, $\mu(o,t)$, $k(o,t)$

Pour une pluie tombée $o$ (mm sur 24 h) et une échéance $t$ (en jours), la prévision $P$ vaut 0 (« sec ») avec la probabilité $p_0$, sinon suit une gamma de moyenne $\mu$ et de forme $k$ ($\theta = \mu/k$). Avec $\ell = \ln o$ :

**si $o \ge 0{,}1$ mm**
$$
\operatorname{logit} p_0 = \alpha \cdot z, \qquad \ln \mu = \beta \cdot z, \qquad \ln k = \gamma \cdot z, \qquad z = (1,\; \ell,\; \ell^2,\; t,\; t\,\ell,\; t^2)
$$
**si $o < 0{,}1$ mm** (rien n'est tombé) : mêmes formes avec $z = (1,\; t,\; t^2)$.

Les formes viennent de ce qu'on a vu : lois de puissance en $o$ (droites en log-log), avec un terme $\ell^2$ pour la courbure des fortes pluies, et des paramètres qui varient régulièrement avec $t$. Choix fait sur la vraisemblance de l'année de vérification : sans $\ell^2$ −25 254, avec −25 065 ; ajouter $t\,\ell^2$ ou des termes en $\ln t$ ne gagne que 7 à 13. Ajustement par **maximum de vraisemblance** sur les années d'apprentissage : régression logistique pour $p_0$ ; pour $(\beta, \gamma)$, on minimise
$$
-\sum_{i \,:\, P_i \ge 0{,}1} \ln f\!\left(P_i\,;\,k_i,\; \mu_i / k_i\right)
$$
(densité gamma, `scipy.optimize.minimize`). Les deux ajustements sont indépendants car la vraisemblance se sépare en « sec ou non » et « combien ».

In [ ]:
from scipy import optimize, special
from sklearn.linear_model import LogisticRegression

def z_pluie(o, j):
    l = np.log(np.maximum(o, SEC))
    return np.column_stack([np.ones_like(l), l, l ** 2, j, j * l, j ** 2])

def z_sec(j):
    return np.column_stack([np.ones_like(j), j, j ** 2])

def nll_gamma(par, Z, y):
    n = Z.shape[1]
    mu, k = np.exp(Z @ par[:n]), np.exp(Z @ par[n:])
    return -np.sum(stats.gamma.logpdf(y, k, scale=mu / k))

def ajuster_modele(d):
    m = {}
    for nom, sous, Z in (("pluie", d[d["o"] >= SEC], lambda x: z_pluie(x["o"].values, x["jours"].values)),
                         ("sec", d[d["o"] < SEC], lambda x: z_sec(x["jours"].values))):
        m[f"p0_{nom}"] = LogisticRegression(C=1e6, max_iter=2000).fit(Z(sous)[:, 1:], (sous["p"] < SEC).astype(int))
        h = sous[sous["p"] >= SEC]
        x0 = np.zeros(2 * Z(h).shape[1]); x0[0] = np.log(h["p"].mean())
        r = optimize.minimize(nll_gamma, x0, args=(Z(h), h["p"].values), method="L-BFGS-B")
        m[f"g_{nom}"] = r.x
        print(f"{nom:5s} : {len(sous):,} cas, gamma ajustée sur {len(h):,} (convergé : {r.success})")
    return m

def parametres(m, o, j):
    """(p0, mu, k) du modèle pour des tableaux o (mm sur 24 h) et j (jours)."""
    o, j = np.asarray(o, float), np.asarray(j, float)
    seche = o < SEC
    Zp, Zs = z_pluie(o, j), z_sec(j)
    p0 = np.where(seche, m["p0_sec"].predict_proba(Zs[:, 1:])[:, 1], m["p0_pluie"].predict_proba(Zp[:, 1:])[:, 1])
    gp, gs = m["g_pluie"], m["g_sec"]
    mu = np.where(seche, np.exp(Zs @ gs[:3]), np.exp(Zp @ gp[:6]))
    k = np.where(seche, np.exp(Zs @ gs[3:]), np.exp(Zp @ gp[6:]))
    return p0, mu, k

MODELE = ajuster_modele(D[~D["test"]])

**Le modèle reproduit-il la dépendance en $t$ ?** Pour quatre classes de pluie tombée, on compare, échéance par échéance et sur les années d'apprentissage, la part de secs, la moyenne et l'écart-type de la partie non sèche, mesurés (points) et donnés par le modèle (traits ; moyenne sur les mêmes cas des $\mu_i$, et écart-type du mélange des gammas).

In [ ]:
CLASSES_O = [("rien (< 0,1 mm)", 0, SEC), ("1 à 2 mm", 1, 2), ("5 à 10 mm", 5, 10), ("20 à 40 mm", 20, 40)]

def comparer(d, m):
    lignes = []
    for nom, a, b in CLASSES_O:
        for t, x in d[(d["o"] >= a) & (d["o"] < b)].groupby("t"):
            p0, mu, k = parametres(m, x["o"].values, x["jours"].values)
            hum = x["p"] >= SEC
            var_m = mu ** 2 / k
            moy_m = np.average(mu, weights=1 - p0)
            sd_m = np.sqrt(np.average(var_m + mu ** 2, weights=1 - p0) - moy_m ** 2)
            lignes.append(dict(classe=nom, jours=t / 24, sec_mesure=1 - hum.mean(), sec_modele=p0.mean(),
                               moy_mesure=x.loc[hum, "p"].mean(), moy_modele=moy_m, sd_mesure=x.loc[hum, "p"].std(), sd_modele=sd_m))
    return pd.DataFrame(lignes)

cmp = comparer(D[~D["test"]], MODELE)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
fig.suptitle("Prévision de pluie sur 24 h selon l'échéance : mesures (points) et modèle (traits), années d'apprentissage")
for (nom, _, _), couleur in zip(CLASSES_O, ["C7", "C0", "C1", "C3"]):
    x = cmp[cmp["classe"] == nom]
    for ax, q, titre in zip(axes, ["sec", "moy", "sd"], ["part de prévisions sèches", "moyenne de la partie non sèche (mm)", "écart-type de la partie non sèche (mm)"]):
        ax.plot(x["jours"], x[f"{q}_mesure"], "o", color=couleur, label=f"tombé : {nom}")
        ax.plot(x["jours"], x[f"{q}_modele"], "-", color=couleur)
        ax.set_title(titre); ax.set_xlabel("échéance (jours)")
axes[0].set_ylabel("probabilité"); axes[1].set_ylabel("mm sur 24 h"); axes[2].set_ylabel("mm sur 24 h")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 9. Vérification sur la dernière année (jamais vue par le modèle)

**Histogrammes.** Échéances 1, 3 et 6 jours (lignes) × classes de pluie tombée (colonnes) : histogramme des vraies prévisions non sèches, et densité du modèle (moyenne, sur les mêmes cas, de leurs densités gamma). Le titre compare la part de secs mesurée et prévue par le modèle.

In [ ]:
TEST = D[D["test"]]
ECH_VERIF = [24, 72, 144]
fig, axes = plt.subplots(len(ECH_VERIF), len(CLASSES_O), figsize=(17, 11))
fig.suptitle("Dernière année : vraies prévisions de pluie sur 24 h (barres) et modèle (courbe rouge)", fontsize=13)
for i, t in enumerate(ECH_VERIF):
    for j, (nom, a, b) in enumerate(CLASSES_O):
        ax = axes[i, j]
        x = TEST[(TEST["t"] == t) & (TEST["o"] >= a) & (TEST["o"] < b)]
        p0, mu, k = parametres(MODELE, x["o"].values, x["jours"].values)
        hum = x.loc[x["p"] >= SEC, "p"]
        if len(hum) < 15:
            ax.set_title(f"{t // 24} j, tombé {nom} : trop peu ({len(x)})", fontsize=9); continue
        haut = hum.quantile(0.99)
        ys = np.linspace(SEC, haut, 200)
        dens = np.average([stats.gamma.pdf(ys, kk, scale=mm / kk) for kk, mm in zip(k, mu)], axis=0, weights=1 - p0)
        ax.hist(hum, bins=30, range=(0, haut), density=True, alpha=0.5, label=f"{len(hum)} vraies prévisions non sèches")
        ax.plot(ys, dens, "C3", lw=2, label="modèle")
        ax.set_title(f"échéance {t // 24} j, tombé {nom}\nsecs : mesuré {1 - len(hum) / len(x):.1%}, modèle {p0.mean():.1%}", fontsize=9)
        ax.set_xlabel("pluie prévue (mm sur 24 h)"); ax.set_ylabel("densité (1/mm)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Calibration : PIT.** Pour chaque cas, $u = F(P_{\text{vrai}})$ où $F$ est la fonction de répartition du modèle,
$$
F(y) = p_0 + (1-p_0)\,F_\Gamma(y\,;k, \mu/k) \quad (y \ge 0,1), \qquad u \sim \mathcal{U}(0, p_0) \;\text{ si } P_{\text{vrai}} < 0{,}1 .
$$
Si le modèle est juste, $u$ est **uniforme** entre 0 et 1 : histogramme plat à 1. Bosse au milieu = modèle trop large ; creux au milieu (bords hauts) = trop étroit ; pente = biais. À droite : $p_0$ prévu par le modèle contre la part de secs vraiment observée (diagramme de fiabilité) ; sur la diagonale = juste.

In [ ]:
rng = np.random.default_rng(0)
p0, mu, k = parametres(MODELE, TEST["o"].values, TEST["jours"].values)
pv = TEST["p"].values
u = np.where(pv < SEC, rng.uniform(0, p0), p0 + (1 - p0) * stats.gamma.cdf(pv, k, scale=mu / k))

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle("Dernière année : calibration du modèle")
for ax, (a, b) in zip(axes[:3], [(1, 2), (3, 5), (6, 9)]):
    sel = (TEST["jours"] >= a).values & (TEST["jours"] <= b).values
    ax.hist(u[sel], bins=10, range=(0, 1), density=True, alpha=0.6, edgecolor="white")
    ax.axhline(1, color="k", ls="--", lw=1, label="modèle parfait")
    ax.set_ylim(0, 1.6); ax.set_title(f"PIT, échéances {a} à {b} j ({sel.sum():,} cas)")
    ax.set_xlabel("u = F(vraie prévision)"); ax.set_ylabel("densité"); ax.legend(fontsize=8)
classes_p0 = pd.cut(p0, np.linspace(0, 1, 11))
fiab = pd.DataFrame({"prevu": p0, "observe": pv < SEC}).groupby(classes_p0, observed=True).mean()
axes[3].plot(fiab["prevu"], fiab["observe"], "o-", label="par tranche de p0")
axes[3].plot([0, 1], [0, 1], "k--", lw=1, label="parfait")
axes[3].set_xlabel("probabilité de sec donnée par le modèle"); axes[3].set_ylabel("part de secs observée")
axes[3].set_title("fiabilité de la probabilité de sec"); axes[3].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10. Bilan du modèle d'erreur de la pluie (cumuls de 24 h, échéances 1 à 9 jours)

Pour une pluie tombée $o$ et une échéance $t$, la prévision vaut 0 avec la probabilité $p_0(o,t)$, sinon suit une gamma de moyenne $\mu(o,t)$ et de forme $k(o,t)$ ; $\operatorname{logit} p_0$, $\ln \mu$, $\ln k$ sont des fonctions simples de $(\ln o, (\ln o)^2, t, t \ln o, t^2)$, ajustées par maximum de vraisemblance.

**Vérifié sur l'année jamais vue (octobre 2025 – octobre 2026) :**
- **probabilité de sec** : fiable à toutes les échéances (diagramme de fiabilité sur la diagonale, écarts ≤ 0,04) ;
- **calibration** : PIT plat de 3 à 9 jours (barres entre 0,89 et 1,15) ; à 1–2 jours, petite bosse au milieu (0,82 à 1,26) : le modèle y est **un peu trop large**, surtout pour les fortes pluies (écart-type 11,5 mm au lieu de 8,6 pour 20–40 mm tombés à 1 jour) ;
- **moyennes** : justes, y compris pour les fortes pluies grâce au terme $(\ln o)^2$ (25,0 mm pour 24,9 à 1 jour).

**Limites :** une seule année de vérification, sur les 7 carreaux complets ; chaque cumul est traité seul : rien encore sur le fait qu'une erreur touche **tous les carreaux d'un même épisode** et **plusieurs jours de suite** — indispensable avant d'abîmer la météo d'entraînement.

## 11. Les lois, explicitement

Pluie tombée $o$ en mm sur 24 h, échéance $t$ en jours (de 1 à 9), $\ell = \ln o$. La prévision $P$ (mm sur 24 h) est fabriquée ainsi :
$$
P = \begin{cases} 0 & \text{avec la probabilité } p_0 \\[2pt] \sim \Gamma\!\left(k,\; \theta = \mu / k\right) & \text{sinon} \end{cases}
\qquad\text{avec}\qquad p_0 = \frac{1}{1 + e^{-\lambda}} .
$$

**Si $o \ge 0{,}1$ mm :**
$$
\begin{aligned}
\lambda &= -2{,}724 \;-\; 1{,}181\,\ell \;-\; 0{,}0677\,\ell^2 \;+\; 0{,}426\,t \;+\; 0{,}114\,t\,\ell \;-\; 0{,}0180\,t^2 \\
\ln \mu &= \;\;\,0{,}173 \;+\; 0{,}707\,\ell \;+\; 0{,}0749\,\ell^2 \;+\; 0{,}177\,t \;-\; 0{,}0784\,t\,\ell \;-\; 0{,}0033\,t^2 \\
\ln k &= \;\;\,0{,}564 \;+\; 0{,}363\,\ell \;+\; 0{,}0380\,\ell^2 \;-\; 0{,}224\,t \;-\; 0{,}0476\,t\,\ell \;+\; 0{,}0111\,t^2
\end{aligned}
$$

**Si $o < 0{,}1$ mm (rien n'est tombé) :**
$$
\begin{aligned}
\lambda &= \;\;\,2{,}872 \;-\; 0{,}287\,t \;+\; 0{,}0016\,t^2 \\
\ln \mu &= -1{,}463 \;+\; 0{,}559\,t \;-\; 0{,}0293\,t^2 \\
\ln k &= -0{,}146 \;-\; 0{,}0192\,t \;-\; 0{,}0041\,t^2
\end{aligned}
$$

Exemple : $o = 5$ mm, $t = 3$ jours $\Rightarrow$ $p_0 = 4{,}2\,\%$, $\mu = 5{,}09$ mm, $k = 1{,}56$, $\theta = 3{,}26$ mm.

Valable pour : cumuls de 24 h, échéances de 1 à 9 jours, prévisions ECMWF IFS 9 km émises à 0 h UTC, La Clapière ; ajusté sur mars 2024 – septembre 2025. Les nombres ci-dessus sont ceux de cet ajustement ; la cellule suivante les recalcule à partir de `MODELE` (ils changent si on réajuste).

In [ ]:
COEFFICIENTS = {"p0_pluie": [float(MODELE["p0_pluie"].intercept_[0]), *map(float, MODELE["p0_pluie"].coef_[0])],
                "p0_sec": [float(MODELE["p0_sec"].intercept_[0]), *map(float, MODELE["p0_sec"].coef_[0])],
                "g_pluie": list(map(float, MODELE["g_pluie"])), "g_sec": list(map(float, MODELE["g_sec"]))}
termes_pluie, termes_sec = ["1", "ℓ", "ℓ²", "t", "t·ℓ", "t²"], ["1", "t", "t²"]
pd.concat({
    "o ≥ 0,1 mm": pd.DataFrame({"λ (logit p0)": COEFFICIENTS["p0_pluie"], "ln μ": COEFFICIENTS["g_pluie"][:6],
                                "ln k": COEFFICIENTS["g_pluie"][6:]}, index=termes_pluie),
    "o < 0,1 mm": pd.DataFrame({"λ (logit p0)": COEFFICIENTS["p0_sec"], "ln μ": COEFFICIENTS["g_sec"][:3],
                                "ln k": COEFFICIENTS["g_sec"][3:]}, index=termes_sec),
}).round(4)

## 12. Température : moyennes sur 24 h

Même découpage que pour la pluie : pour chaque carreau, prévision émise à $r$ et échéance $t$ (1 à 9 jours), la **moyenne** de la température prévue et de la température ERA5-Land sur les 24 h qui finissent à $r+t$. On fait confiance à Open-Meteo : pas de correction de biais, seulement une erreur aléatoire autour de la température observée,
$$
T_{\text{prévue}} = T_{\text{obs}} + \varepsilon, \qquad \sigma = \operatorname{sd}(\varepsilon) \text{ fonction de l'échéance et de la saison.}
$$

In [ ]:
from matplotlib.colors import LogNorm
from scipy import stats

prevT = pd.read_parquet(RUNS, columns=["carreau", "run", "time", "temperature_2m"])
prevT["echeance"] = ((prevT["time"] - prevT["run"]) / pd.Timedelta("1h")).astype(int)
obsT = pd.concat({c: pd.read_parquet(OBS / f"{c}_horaire.parquet", columns=["temperature_2m"])["temperature_2m"] for c in carreaux},
                 names=["carreau", "time"])
prevT["obs"] = obsT.reindex(pd.MultiIndex.from_arrays([prevT["carreau"], prevT["time"]])).to_numpy()
prevT = prevT.dropna()

def moyennes_T(t, pas=24):
    f = prevT[(prevT["echeance"] > t - pas) & (prevT["echeance"] <= t)]
    g = f.groupby(["carreau", "run"]).agg(prevu=("temperature_2m", "mean"), obs=("obs", "mean"), n=("obs", "size"))
    return g[g["n"] == pas].reset_index().assign(t=t)

DT = pd.concat([moyennes_T(t) for t in range(24, 240, 24)], ignore_index=True)
DT["jours"] = DT["t"] / 24
DT["mois"] = (DT["run"] + pd.to_timedelta(DT["t"], unit="h")).dt.month
print(f"{len(DT):,} moyennes de 24 h")

x = DT[DT["t"] == 72]
fig, ax = plt.subplots(figsize=(6.5, 6))
_, _, _, im = ax.hist2d(x["obs"], x["prevu"], bins=60, norm=LogNorm(), cmap="turbo")
lim = [x[["obs", "prevu"]].min().min(), x[["obs", "prevu"]].max().max()]
ax.plot(lim, lim, "k--", lw=1, label="prévision parfaite")
ax.set_xlabel("température observée, moyenne sur 24 h (°C)"); ax.set_ylabel("température prévue 3 jours avant (°C)")
ax.set_title(f"Température à 3 jours ({len(x):,} journées)"); ax.legend(loc="upper left")
fig.colorbar(im, ax=ax, label="nombre de journées (échelle log)"); plt.tight_layout(); plt.show()
print(f"biais moyen {np.mean(x['prevu'] - x['obs']):+.2f} °C   écart-type de l'erreur {np.std(x['prevu'] - x['obs']):.2f} °C   corrélation {np.corrcoef(x['obs'], x['prevu'])[0, 1]:.3f}")

**Selon l'échéance.** $\varepsilon = T_{\text{prévue}} - T_{\text{obs}}$ : sa moyenne (pour vérifier qu'elle est proche de 0) et son écart-type $\sigma$.

In [ ]:
eps = DT["prevu"] - DT["obs"]
parT = DT.assign(eps=eps).groupby("jours")["eps"].agg(moyenne="mean", sigma="std", asymetrie=stats.skew,
                                                     aplatissement=lambda e: stats.kurtosis(e))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Température moyenne sur 24 h : erreur ε = T_prévue − T_obs selon l'échéance")
axes[0].plot(parT.index, parT["sigma"], "o-"); axes[0].set_title("écart-type σ de ε")
axes[0].set_xlabel("échéance (jours)"); axes[0].set_ylabel("°C")
axes[1].plot(parT.index, parT["moyenne"], "o-"); axes[1].axhline(0, color="k", ls="--", lw=1)
axes[1].set_title("moyenne de ε (proche de 0 : pas de biais à corriger)"); axes[1].set_xlabel("échéance (jours)"); axes[1].set_ylabel("°C")
plt.tight_layout(); plt.show()
parT.round(3)

**L'erreur est-elle gaussienne ?** Histogramme de $\varepsilon$ avec la gaussienne de même écart-type, et graphique quantile-quantile (points sur la diagonale = gaussienne). Asymétrie et aplatissement en excès valent 0 pour une gaussienne.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
fig.suptitle("Erreur ε = T_prévue − T_obs : gaussienne ?")
for j, t in enumerate([24, 72, 144]):
    x = DT[DT["t"] == t]
    r = x["prevu"] - x["obs"]
    ax = axes[0, j]
    ax.hist(r, bins=60, density=True, alpha=0.5, label="erreur ε")
    xs = np.linspace(r.min(), r.max(), 200)
    ax.plot(xs, stats.norm.pdf(xs, 0, r.std()), "C3", lw=2, label=f"gaussienne σ = {r.std():.2f} °C")
    ax.set_title(f"échéance {t // 24} j : asymétrie {stats.skew(r):+.2f}, aplatissement {stats.kurtosis(r):+.2f}")
    ax.set_xlabel("erreur ε (°C)"); ax.set_ylabel("densité (1/°C)"); ax.legend(fontsize=8)
    stats.probplot(r, dist="norm", plot=axes[1, j])
    axes[1, j].set_title(f"quantile-quantile, échéance {t // 24} j"); axes[1, j].set_xlabel("quantiles d'une gaussienne")
    axes[1, j].set_ylabel("quantiles de ε (°C)")
plt.tight_layout(); plt.show()

**Selon la saison** (à 3 jours) : moyenne et écart-type de $\varepsilon$ par mois de l'heure prévue.

In [ ]:
x = DT[DT["t"] == 72]
saisonT = (x["prevu"] - x["obs"]).groupby(x["mois"]).agg(moyenne="mean", sigma="std", cas="size")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Température à 3 jours selon le mois : erreur ε = T_prévue − T_obs")
axes[0].bar(saisonT.index, saisonT["sigma"]); axes[0].set_title("écart-type σ (°C)")
axes[1].bar(saisonT.index, saisonT["moyenne"]); axes[1].axhline(0, color="k", lw=1); axes[1].set_title("moyenne (°C)")
for ax in axes:
    ax.set_xlabel("mois"); ax.set_xticks(range(1, 13))
plt.tight_layout(); plt.show()
saisonT.round(3)

**Ce qu'on voit (température moyenne sur 24 h).**
- **Pas de biais à corriger** : la moyenne de $\varepsilon$ reste entre −0,24 et −0,08 °C à toutes les échéances.
- **La dispersion grandit avec l'échéance** : $\sigma$ = 0,6 °C à 1 jour, 0,9 à 3 jours, 1,3 à 5 jours, 2,1 à 7 jours, 2,9 à 9 jours.
- **L'erreur n'est pas gaussienne aux courtes échéances** : pic pointu et longue queue du côté froid à 1 jour ; elle s'en rapproche vers 5–6 jours, sauf aux extrémités. Une loi à queues plus lourdes (Student) conviendra mieux à courte échéance.
- **La saison compte** : $\sigma$ plus faible de juillet à octobre qu'au printemps.

**À faire ensuite :** écrire $\sigma(t, \text{mois})$, choisir la loi des queues, puis mesurer si l'erreur est la même sur tous les carreaux et plusieurs jours de suite.

## Annexe : page interactive

La même analyse que la partie 3, en direct dans un navigateur : sens, pas du cumul, début du cumul (heures après l'émission), fourchette de la condition (début + taille) ou condition sèche. Avec le modèle de la partie 8, la page trace aussi sa densité (tirets verts), à comparer à la gamma ajustée sur les cas affichés. Page autonome, à ouvrir dans Safari (pas dans l'aperçu de VS Code).

In [ ]:
from src.graphes import explorer_pluie
CARREAUX_COMPLETS = [835931, 841112, 838522, 830767, 846307, 841113, 833346]     # les 7 choisis, prévisions complètes
explorer_pluie(CARREAUX_COMPLETS, "../data/outputs/erreurs_pluie.html", modele=COEFFICIENTS)